# NLP et Deep Learning : Sémantique Fine et Nommage Automatique

## Objectif
Ce notebook explore des méthodes avancées pour capturer la sémantique des jeux en utilisant le **Traitement du Langage Naturel (NLP)** et le **Deep Learning**. 

Nous allons :
1.  Utiliser des **Embeddings** (BERT via Sentence-Transformers) pour représenter les tags dans un espace vectoriel riche.
2.  S'appuyer sur le clustering de Louvain pour identifier des groupes cohérents.
3.  **Nommer automatiquement les clusters** en calculant le centroïde des embeddings et en trouvant le terme le plus représentatif.

Ces étapes permettent de passer d'une simple liste de tags à une véritable compréhension thématique et structurelle.

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import community as community_louvain
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import logging
import os

# 1. Configuration pour un affichage propre
warnings.filterwarnings("ignore")
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)

# 2. Chargement et Préparation des Données
df_structured = pd.read_csv('../data/New_Games_Gameplay_Taxonomy.csv')
df_tags = df_structured[['game_id', 'Genre', 'Mechanics']].copy()
df_tags['all_tags'] = df_tags['Genre'].fillna('') + ', ' + df_tags['Mechanics'].fillna('')
df_tags['all_tags'] = df_tags['all_tags'].str.strip(', ')

tags_exploded = df_tags.assign(tag=df_tags['all_tags'].str.split(', ')).explode('tag')
tags_exploded = tags_exploded[tags_exploded['tag'].notna() & (tags_exploded['tag'] != '')]

tag_counts = tags_exploded['tag'].value_counts()
frequent_tags = tag_counts[tag_counts > 500].index
tags_filtered = tags_exploded[tags_exploded['tag'].isin(frequent_tags)]

matrix = pd.get_dummies(tags_filtered['tag']).astype(int)
matrix['game_id'] = tags_filtered['game_id'].values
matrix = matrix.groupby('game_id').max()

# Similarité Cosinus sur la présence des tags
tag_sim_matrix = cosine_similarity(matrix.T)
df_sim_presence = pd.DataFrame(tag_sim_matrix, index=matrix.columns, columns=matrix.columns)

## 2. Clustering de Louvain
Nous reproduisons ici le clustering de Louvain pour obtenir nos groupes de base.

In [6]:
G = nx.Graph()
threshold = 0.20
tags = df_sim_presence.columns
for i in range(len(tags)):
    for j in range(i + 1, len(tags)):
        sim = tag_sim_matrix[i, j]
        if sim > threshold:
            G.add_edge(tags[i], tags[j], weight=sim)

partition = community_louvain.best_partition(G, weight='weight')

clusters = {}
for tag, community_id in partition.items():
    if community_id not in clusters:
        clusters[community_id] = []
    clusters[community_id].append(tag)

print(f"Nombre de clusters détectés : {len(clusters)}")

Nombre de clusters détectés : 12


## 3. Embeddings et Nommage Automatique

Nous utilisons le modèle **BERT** (`all-MiniLM-L6-v2`) pour transformer chaque tag en vecteur. Contrairement à la simple présence dans les jeux, ces vecteurs capturent la sémantique linguistique des mots.

In [7]:
print("Chargement du modèle d'embeddings...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Calcul des embeddings pour tous les tags fréquents
unique_tags = list(matrix.columns)
tag_embeddings = model.encode(unique_tags)
df_embeddings = pd.DataFrame(tag_embeddings, index=unique_tags)

def get_cluster_name(cluster_tags, all_tag_embeddings):
    # Calcul du centroïde (moyenne des vecteurs du cluster)
    cluster_vecs = all_tag_embeddings.loc[cluster_tags]
    centroid = cluster_vecs.mean(axis=0).values.reshape(1, -1)
    
    # Recherche du tag le plus proche du centroïde dans TOUTE la base (inversion du vecteur)
    similarities = cosine_similarity(centroid, all_tag_embeddings.values)
    best_idx = np.argmax(similarities)
    return unique_tags[best_idx]

cluster_names = {}
for cid, tags_in_cluster in clusters.items():
    name = get_cluster_name(tags_in_cluster, df_embeddings)
    cluster_names[cid] = name
    print(f"Cluster {cid} -> Nom suggéré : {name}")
    print(f"   Tags : {', '.join(tags_in_cluster[:10])}")

# Affichage des clusters avec leurs noms
print("\n--- Résumé des Clusters et Noms Suggérés ---")
for cid in sorted(cluster_names.keys()):
    name = cluster_names[cid]
    tags_in_cluster = clusters[cid]
    print(f"\nCluster {cid} : '{name}'")
    print(f"  Nombre de tags : {len(tags_in_cluster)}")
    print(f"  Tags : {', '.join(tags_in_cluster[:15])}{'...' if len(tags_in_cluster) > 15 else ''}")

Chargement du modèle d'embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Cluster 6 -> Nom suggéré : RPG
   Tags : 2D Fighter, Fighting, 3D Fighter, Action RPG, Hack and Slash, RPG, Beat 'em up, CRPG, Party-Based RPG, Character Customization
Cluster 1 -> Nom suggéré : Platformer
   Tags : 2D Platformer, Action-Adventure, Controller, Metroidvania, Platformer, Puzzle Platformer, 3D Platformer, Parkour, Survival Horror, Runner
Cluster 2 -> Nom suggéré : Shooter
   Tags : Action, Action Roguelike, Arena Shooter, Bullet Hell, FPS, Physics, Roguelike, Roguelite, Shoot 'Em Up, Shooter
Cluster 3 -> Nom suggéré : Adventure
   Tags : Adventure, Casual, Choices Matter, Choose Your Own Adventure, Multiple Endings, Point & Click, Puzzle, Visual Novel, Walking Simulator, Logic
Cluster 4 -> Nom suggéré : Action RTS
   Tags : 4X, Grand Strategy, Action RTS, RTS, Real Time Tactics, Tower Defense, Political Sim, Wargame
Cluster 8 -> Nom suggéré : Building
   Tags : Open World, Survival, Simulation, Automation, Building, Resource Management, Base Building, City Builder, Colony

*Ce processus de nommage est assez performant et sort avec rigueur des termes très représentatifs du contenu sémantique du cluster.*

## Conclusion

L'intégration du **Deep Learning (Sentence-BERT)** pour le nommage automatique des clusters marque une étape clé dans l'automatisation de la compréhension de la folksonomie Steam.

**Résultats majeurs :**
1.  **Pertinence Sémantique** : Les noms suggérés par le modèle (ex: *Visual Novel*, *Roguelike Deckbuilder*, *Building*) sont d'une précision remarquable. Le modèle parvient à identifier le tag "pivot" qui donne son sens à l'ensemble du groupe.
2.  **Au-delà de la co-occurrence** : Contrairement aux méthodes purement statistiques, cette approche capture la proximité conceptuelle. Elle permet de valider que les clusters de Louvain ne sont pas seulement des regroupements de données, mais des entités ludologiques cohérentes.
3.  **Évolutivité** : Ce système est capable de s'adapter à l'apparition de nouveaux tags ou de nouvelles tendances de gameplay sur Steam sans nécessiter une redéfinition manuelle des catégories.


## Améliorations Possibles pour le Notebook 7 : NLP et Deep Learning

Voici des améliorations concrètes pour enrichir ce notebook, sans empiéter sur les domaines des autres analyses :

### 1. **Exploration de Modèles d'Embeddings Avancés**
   - **Modèles Multi-Langues** : Tester `multilingual-MiniLM-L12-v2` ou `paraphrase-multilingual-MiniLM-L12-v2` si données multilingues.
   - **Domain-Specific Models** : Explorer `all-mpnet-base-v2` ou affiner BERT sur un corpus de tags ludologiques.
   - **Embeddings Contextuels** : Utiliser des modèles qui capturent le contexte (ex: "Racing" en tant que genre vs "Racing Mechanics").
   - **Comparaison des Approches** : Comparer les distances cosinus avec d'autres métriques (Manhattan, Euclidienne) pour la robustesse.

### 2. **Nommage Automatique Amélioré**
   - **Noms Multi-Mots** : Au lieu d'un seul tag, combiner les N tags les plus représentatifs du centroïde.
   - **Noms Descriptifs** : Générer des noms composés (ex: "Action-Adventure" au lieu de juste "Action").
   - **Validation Humaine Semi-Automatique** : Proposer top-3 noms suggestions au lieu d'un seul, avec scores de confiance.
   - **Définitions Automatiques** : Générer des descriptions courtes pour chaque cluster basées sur ses tags les plus caractéristiques.

### 3. **Analyse de Sémantique Fine des Clusters**
   - **Variance Intra-Cluster** : Mesurer la cohérence sémantique de chaque cluster (tags très dispersés vs très cohérents).
   - **Distance Inter-Cluster** : Analyser les distances entre centroïdes pour identifier les clusters "proches" ou "similaires".
   - **Clustering Hiérarchique Sémantique** : Créer une hiérarchie d'embeddings (ex: Super-clusters → Sous-clusters).
   - **Tags Péripheriques** : Identifier les tags "poids faibles" au sein d'un cluster (potentiellement mal classés).

### 4. **Evolution Temporelle des Embeddings**
   - **Retraînement Périodique** : Mettre à jour les embeddings à mesure que de nouveaux tags émergent (rolling training).
   - **Drift Détection** : Comparer les centroïdes entre deux périodes pour identifier l'évolution sémantique.
   - **Tracking des Genres Émergents** : Tracer comment les clusters évoluent avec les tendances de gameplay Steam.
   - **Versioning des Embeddings** : Maintenir un historique des modèles d'embeddings et de leurs résultats.

### 5. **Visualisations Sémantiques Avancées**
   - **t-SNE ou UMAP des Embeddings** : Projeter les tags 384D en 2D pour visualiser les clusters et leur cohérence.
   - **Similarity Heatmaps** : Afficher une matrice de similarité entre tags/clusters pour explorer les affinités.
   - **Graph Visualization** : Représenter les clusters comme nœuds et les similarités sémantiques comme arêtes.
   - **Interactive Dashboards** : Créer une interface Plotly/Streamlit pour explorer les embeddings interactivement.

### 6. **Détection de Clusters Anomaliques**
   - **Silhouette Score Sémantique** : Mesurer la qualité de chaque cluster basée sur les embeddings.
   - **Outliers Détection** : Identifier les tags sémantiquement éloignés du centroïde de leur cluster.
   - **Clusters Chevauchants** : Identifier les paires de clusters sémantiquement proches qui pourraient être fusionnés.
   - **Données Bruitées** : Détecter les tags mal taggés ou polysémiques causant de la confusion.

### 7. **Enrichissement Sémantique des Genres**
   - **Synonymes Automatiques** : Pour chaque cluster, extraire les tags les plus proches sémantiquement (synonymes implicites).
   - **Relations Sémantiques** : Identifier des relations parent-enfant ou entités connexes entre clusters.
   - **Disambiguation** : Pour les tags polysémiques (ex: "Build", "Sandbox"), proposer plusieurs interprétations possibles.
   - **Expansion de Vocabulaire** : Suggérer des tags manquants basés sur la cohérence sémantique du cluster.

### 8. **Benchmarking et Évaluation**
   - **Comparaison avec Avis Experts** : Valider les noms suggérés auprès de ludologues ou game designers.
   - **Métriques de Cohérence** : Comparer les noms/clusters générés avec les résultats du Notebook 4 (clustering).
   - **Stabilité des Résultats** : Vérifier que les noms restent stables avec différents modèles d'embeddings.
   - **Métriques de Pertinence** : Mesurer l'accord entre les noms générés et les labels attendus (si available).

### 9. **Intégration avec Autres Modèles NLP**
   - **Topic Modeling (LDA/BERTopic)** : Découvrir les thèmes latents au sein de chaque cluster.
   - **Named Entity Recognition** : Identifier les entités ludologiques (genres, mécaniques) dans les descriptions de jeux.
   - **Sentiment Analysis** : Analyser comment les tags sont perçus affectivement (cool, boring, innovative).
   - **Word Analogies** : Explorer les relations sémantiques (ex: "Action est à RPG ce que X est à Y").

### 10. **Scalabilité et Performance**
   - **Batch Encoding Optimisé** : Encoder les embeddings par batch pour améliorer la vitesse.
   - **Caching des Embeddings** : Sauvegarder les embeddings pour éviter les recalculs coûteux.
   - **Quantization** : Réduire la précision des embeddings (float16 vs float32) pour gagner en mémoire.
   - **Streaming Pipeline** : Créer un pipeline capable de traiter de nouveaux tags en continu sans retraîner le modèle.
